# Seed Traceability Status

Config-level audit of seed coverage used by the paper notebooks. Each row is one dataset/probe/model/backbone/layer configuration from the canonical seed manifests, checked against the active seed loader policy: seed 42 plus seeds 101 and 102 must be available, and attentive seeds 101/102 must reach the configured max epoch directly or through the approved continuation audit.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / 'run.py').exists() and (path / 'results').exists():
            return path
    raise RuntimeError('Could not find repo root')

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
from seed_variance_loader import build_seed_traceability_status

RESULTS_DIR = REPO_ROOT / 'results'
OUT_DIR = RESULTS_DIR / 'seed_runs'
DATASET_LABEL = {'mvp': 'MVP', 'intphys2': 'IntPhys2'}
PROBE_LABEL = {'linear': 'Linear', 'mlp': 'MLP', 'temporal_attn': 'Attentive'}


In [ ]:
trace = build_seed_traceability_status(RESULTS_DIR)
trace['Dataset'] = trace['dataset'].map(DATASET_LABEL).fillna(trace['dataset'])
trace['Probe'] = trace['probe'].map(PROBE_LABEL).fillna(trace['probe'])
trace['Model'] = trace['model_label']
trace['Loaded seeds (x/3)'] = trace['loaded_n_seeds'].fillna(0).astype(int).astype(str) + '/3'
trace['Complete 3/3'] = trace['status'].eq('OK')
problems = trace[~trace['Complete 3/3']].copy()

status_path = OUT_DIR / 'seed_traceability_status.csv'
problems_path = OUT_DIR / 'seed_traceability_problems.csv'
trace.to_csv(status_path, index=False)
problems.to_csv(problems_path, index=False)
print(f'Wrote {status_path.relative_to(REPO_ROOT)} ({len(trace)} rows)')
print(f'Wrote {problems_path.relative_to(REPO_ROOT)} ({len(problems)} rows)')

if problems.empty:
    print('OK [seed traceability] every canonical config loaded as 3/3 seeds.')
else:
    print(f'ERROR [seed traceability] {len(problems)} canonical configs are not loaded as 3/3 seeds.')
    display(problems[['Dataset', 'Probe', 'Model', 'backbone', 'experiment', 'layer_key', 'Loaded seeds (x/3)', 'loaded_seeds', 'missing_loaded_seeds', 'status', 'manifest']])


In [ ]:
summary = (
    trace.groupby(['Dataset', 'Probe', 'Model', 'backbone', 'experiment'], as_index=False)
    .agg(configs=('config_id', 'nunique'), complete_configs=('Complete 3/3', 'sum'))
)
summary['missing_configs'] = summary['configs'] - summary['complete_configs']
summary['coverage'] = summary['complete_configs'].astype(int).astype(str) + '/' + summary['configs'].astype(int).astype(str)
summary = summary.sort_values(['Dataset', 'Probe', 'Model', 'backbone', 'experiment'])
display(Markdown("## Coverage Summary"))
display(summary[['Dataset', 'Probe', 'Model', 'backbone', 'experiment', 'coverage', 'missing_configs']])


In [ ]:
detail_cols = ['Dataset', 'Probe', 'Model', 'backbone', 'experiment', 'layer', 'layer_label', 'Loaded seeds (x/3)', 'loaded_seeds', 'loaded_seed_source', 'test_primary_mean', 'test_primary_std', 'status']
for dataset in sorted(trace['Dataset'].dropna().unique()):
    for probe in sorted(trace.loc[trace['Dataset'].eq(dataset), 'Probe'].dropna().unique()):
        section = trace[trace['Dataset'].eq(dataset) & trace['Probe'].eq(probe)].copy()
        if section.empty:
            continue
        display(Markdown(f'## {dataset} - {probe}'))
        display(section[detail_cols].sort_values(['Model', 'backbone', 'experiment', 'layer', 'layer_label']))
